# ClusterSC — West Germany application

This notebook implements the ClusterSC comparison for the West Germany application.

Donor clustering is performed using the original `syclib` implementation provided by Rho et al.:
https://github.com/srho1/ClusterSC

The donor trajectories over the full observation period are used for clustering, and the treated unit is assigned to a donor cluster using its pre-intervention trajectory. After donor selection, we fit an OLS synthetic control directly on the selected donors using the original pre-intervention GDP outcomes (1960–1989). The number of clusters, $K$, is selected from the candidate grid by minimizing the pre-intervention RMSE.

**Before running:** install the required Python packages and make sure that `repgermany.dta` is available at `data/repgermany.dta`.

## Imports

In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

from pathlib import Path
import importlib.metadata

import numpy as np
import pandas as pd

from syclib.cluster import ClusterSC
from syclib import SyntheticControl

## 1. User settings

In [2]:
INPUT_DATA_PATH = Path("data/repgermany.dta")
OUTPUT_DIR = Path("results/realdata/ClusterSC")

TARGET_NAME = "West Germany"
PRE_START = 1960
PRE_END = 1989
POST_START = 1990
POST_END = 2003

CANDIDATE_K = [2, 3, 4, 5]
KMEANS_RANDOM_STATE = 123

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data preparation

In [3]:
data = pd.read_stata(INPUT_DATA_PATH)

data["country"] = data["country"].astype(str)
data["year"] = pd.to_numeric(data["year"])
data["gdp"] = pd.to_numeric(data["gdp"])

data = data.loc[
    data["year"].between(PRE_START, POST_END),
    ["country", "year", "gdp"],
].copy()

if data.duplicated(["country", "year"]).any():
    raise ValueError("Duplicate country-year observations found.")

if data[["country", "year", "gdp"]].isna().any().any():
    raise ValueError("Missing values found in the analysis panel.")

panel_df = (
    data
    .pivot(index="country", columns="year", values="gdp")
    .sort_index()
    .sort_index(axis=1)
)

if TARGET_NAME not in panel_df.index:
    raise ValueError("West Germany is missing from the panel.")

years = panel_df.columns.to_numpy()
pre_mask = (years >= PRE_START) & (years <= PRE_END)
post_mask = (years >= POST_START) & (years <= POST_END)

target_data = panel_df.loc[TARGET_NAME]
target_pre = target_data.loc[years[pre_mask]]

donor_data = panel_df.drop(index=[TARGET_NAME])

y_true = target_data.to_numpy(dtype=float)

## 3. Select K by pre-intervention RMSE

In [4]:
best_pre_rmse = np.inf
best_k = None
best_result = None
selection_records = []

for k in CANDIDATE_K:
    try:
        # Cluster donors. The target is assigned to a donor cluster using only
        # its pre-intervention outcome trajectory.
        csc = ClusterSC(donor_data)
        csc.perform_clustering(
            k=k,
            random_state=KMEANS_RANDOM_STATE,
            method="kmeans",
        )

        pred_cluster = csc.predict_target_cluster(
            target_pre.to_frame(name=TARGET_NAME)
        )

        pred_cluster_arr = np.asarray(pred_cluster).ravel()
        if len(pred_cluster_arr) != 1:
            raise ValueError("Unexpected target-cluster output.")

        pred_cluster = int(pred_cluster_arr[0])
        selected = csc.get_donor_group(pred_cluster)

        if selected.shape[0] == 0:
            continue

        # Fit the OLS synthetic control on the original pre-intervention
        # outcomes (1960--1989).
        X_pre = selected.loc[:, years[pre_mask]].T
        y_pre = target_data.loc[years[pre_mask]].to_frame(name=TARGET_NAME)

        cluster_sc = SyntheticControl()
        cluster_sc.fit(
            X_pre,
            y_pre,
            method="ols",
        )

        # Predict the counterfactual trajectory over the full 1960--2003 period.
        X_full = selected.loc[:, years].T
        full_prediction = np.asarray(
            cluster_sc.predict(X_full)
        ).reshape(-1)

        if len(full_prediction) != len(years):
            raise ValueError(
                "Unexpected ClusterSC prediction length: "
                f"{len(full_prediction)}"
            )

        pre_rmse = np.sqrt(
            np.mean(
                (
                    full_prediction[pre_mask]
                    - y_true[pre_mask]
                ) ** 2
            )
        )

        selection_records.append(
            {
                "K": k,
                "selected_cluster": pred_cluster,
                "n_selected_donors": selected.shape[0],
                "pre_RMSE": pre_rmse,
            }
        )

        if np.isnan(pre_rmse):
            continue

        if pre_rmse < best_pre_rmse:
            best_pre_rmse = pre_rmse
            best_k = k
            best_result = {
                "model": cluster_sc,
                "selected_donors": list(selected.index),
                "counterfactual": full_prediction,
            }

    except Exception as exc:
        selection_records.append(
            {
                "K": k,
                "selected_cluster": np.nan,
                "n_selected_donors": np.nan,
                "pre_RMSE": np.nan,
                "error": str(exc),
            }
        )


if best_k is None or best_result is None:
    raise RuntimeError("No valid ClusterSC fit was obtained.")

## 4. Donor weights

In [5]:
selected_donors = best_result["selected_donors"]
selected_weights = np.asarray(
    best_result["model"].model.coef_
).reshape(-1)

if len(selected_weights) != len(selected_donors):
    raise ValueError(
        "Unexpected mismatch between selected donors and weights."
    )

all_donors = list(donor_data.index)

weight_table = pd.DataFrame(
    {
        "country": all_donors,
        "selected": [
            d in selected_donors
            for d in all_donors
        ],
    }
)

weight_map = dict(zip(selected_donors, selected_weights))
weight_table["weight"] = weight_table["country"].map(weight_map)

print(
    f"\nSelected K = {best_k}; "
    f"pre-RMSE = {best_pre_rmse:.4f}"
)
print("Selected donors:", selected_donors)

print("\n=== ClusterSC selected donor weights ===")
print(
    weight_table.loc[weight_table["selected"]]
    .to_string(index=False)
)


Selected K = 2; pre-RMSE = 241.9521
Selected donors: ['Greece', 'New Zealand', 'Portugal', 'Spain']

=== ClusterSC selected donor weights ===
    country  selected    weight
     Greece      True  0.135404
New Zealand      True  0.669500
   Portugal      True  1.051813
      Spain      True -0.176313


## 5. Counterfactual path and ATT

In [6]:
counterfactual = best_result["counterfactual"]

trajectory = pd.DataFrame(
    {
        "year": years,
        "observed": y_true,
        "counterfactual": counterfactual,
    }
)

trajectory["effect"] = (
    trajectory["observed"]
    - trajectory["counterfactual"]
)

att = trajectory.loc[
    trajectory["year"].between(POST_START, POST_END),
    "effect",
].mean()

summary = pd.DataFrame(
    {
        "method": ["ClusterSC"],
        "selected_K": [best_k],
        "pre_RMSE": [best_pre_rmse],
        "post_ATT": [att],
    }
)

selection_table = pd.DataFrame(selection_records)

print("\n=== ClusterSC summary ===")
print(summary.to_string(index=False))

print("\n=== K-selection diagnostics ===")
print(selection_table.to_string(index=False))


=== ClusterSC summary ===
   method  selected_K   pre_RMSE     post_ATT
ClusterSC           2 241.952111 -1814.288718

=== K-selection diagnostics ===
 K  selected_cluster  n_selected_donors   pre_RMSE
 2                 0                  4 241.952111
 3                 0                  4 241.952111
 4                 0                  2 376.056722
 5                 0                  2 376.056722


## 6. Export

In [7]:
weight_table.to_csv(
    OUTPUT_DIR / "ClusterSC_weights.csv",
    index=False,
)

trajectory.to_csv(
    OUTPUT_DIR / "ClusterSC_trajectory.csv",
    index=False,
)

summary.to_csv(
    OUTPUT_DIR / "ClusterSC_summary.csv",
    index=False,
)

selection_table.to_csv(
    OUTPUT_DIR / "ClusterSC_K_selection.csv",
    index=False,
)

with open(
    OUTPUT_DIR / "python_environment.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        "ClusterSC/syclib source repository: "
        "https://github.com/srho1/ClusterSC\n"
    )
    try:
        f.write(
            "syclib installed version: "
            + importlib.metadata.version("syclib")
            + "\n"
        )
    except importlib.metadata.PackageNotFoundError:
        pass
    f.write("numpy: " + np.__version__ + "\n")
    f.write("pandas: " + pd.__version__ + "\n")